In [25]:
import math
import sys
from collections import Counter, defaultdict
import nltk
from nltk.corpus import brown

from nltk import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import spacy
import string
import re
from collections import Counter
from pprint import pprint

nltk.download('brown')

[nltk_data] Downloading package brown to /home/raghav/nltk_data...
[nltk_data]   Package brown is already up-to-date!


True

In [11]:
words = [w.lower() for w in brown.words()]

word_counts = Counter(words)

vocab = {w for w in word_counts if word_counts[w] >= 10}

filtered_words = [w for w in words if w in vocab]
bigram_counts = Counter(zip(filtered_words[:-1], filtered_words[1:]))


unigram_counts = Counter(words)
bigram_counts = Counter(zip(words, words[1:]))

In [30]:
def compute_pmi(w1, w2):
    if (w1, w2) not in bigram_counts:
        return float('-inf')

    p_w1 = word_counts[w1] / N
    p_w2 = word_counts[w2] / N
    p_w1_w2 = bigram_counts[(w1, w2)] / (N - 1)
    
    return math.log2(p_w1_w2 / (p_w1 * p_w2))

def compute_ppmi(w1, w2):
    pmi = compute_pmi(w1, w2)
    return max(0, pmi)


In [13]:
pmi_scores = []
for (w1, w2), count in bigram_counts.items():
    if count > 0:
        pmi_scores.append(((w1, w2), compute_pmi(w1, w2)))

pmi_sorted = sorted(pmi_scores, key=lambda x: x[1], reverse=True)

In [15]:
top_20 = pmi_sorted[:20]
top_20


[(('term-end', 'presentments'), 20.147176349454707),
 (('durwood', 'pye'), 20.147176349454707),
 (('182', 'scholastics'), 20.147176349454707),
 (('decries', 'joblessness'), 20.147176349454707),
 (('audrey', 'knecht'), 20.147176349454707),
 (('vouchers', 'certifying'), 20.147176349454707),
 (('barnet', 'lieberman'), 20.147176349454707),
 (('cedvet', 'sunay'), 20.147176349454707),
 (('submarine-ball', 'hurler'), 20.147176349454707),
 (('6-foot-3-inch', '158-pounder'), 20.147176349454707),
 (("tuttle's", '390-foot'), 20.147176349454707),
 (("forbes's", 'paget'), 20.147176349454707),
 (("hallowell's", 'milties'), 20.147176349454707),
 (('swift-striding', 'jamaican'), 20.147176349454707),
 (('mal', 'whitfield'), 20.147176349454707),
 (("tech's", 'sweat-suits'), 20.147176349454707),
 (("raiders'", '38-7'), 20.147176349454707),
 (('abner', 'haynes'), 20.147176349454707),
 (('patti', 'waggin'), 20.147176349454707),
 (('camilo', 'carreon'), 20.147176349454707)]

In [16]:
bottom_20 = pmi_sorted[-20:]
bottom_20

[(('the', 'have'), -7.892005201409327),
 (('the', 'not'), -8.117844175682542),
 ((',', ';'), -8.127298227499267),
 (('of', 'for'), -8.216990357126361),
 (('of', 'he'), -8.225932866969588),
 (('he', 'of'), -8.225932866969588),
 (('the', 'i'), -8.281566425673033),
 (('of', 'to'), -8.679914634270766),
 (('his', 'the'), -8.719607714287493),
 (('a', 'in'), -8.735419602126678),
 (('the', 'and'), -9.178764350546967),
 (('the', ','), -9.194380023267053),
 (('the', 'is'), -9.250645810292173),
 (('the', 'in'), -9.328362866210027),
 (('and', 'and'), -9.485691072349855),
 (('of', 'of'), -10.15707638123033),
 (('and', '.'), -10.259902885171144),
 (('the', 'a'), -10.44881936610304),
 (('the', '.'), -10.537938664089411),
 (('.', ','), -11.27551855789123)]

In [19]:
ppmi_scores = []
for (w1, w2), count in bigram_counts.items():
    if count > 0:
        ppmi_scores.append(((w1, w2), compute_ppmi(w1, w2)))

ppmi_sorted = sorted(ppmi_scores, key=lambda x: x[1], reverse=True)

In [21]:
top_20_ppmi = ppmi_sorted[:20]
top_20_ppmi

[(('term-end', 'presentments'), 20.147176349454707),
 (('durwood', 'pye'), 20.147176349454707),
 (('182', 'scholastics'), 20.147176349454707),
 (('decries', 'joblessness'), 20.147176349454707),
 (('audrey', 'knecht'), 20.147176349454707),
 (('vouchers', 'certifying'), 20.147176349454707),
 (('barnet', 'lieberman'), 20.147176349454707),
 (('cedvet', 'sunay'), 20.147176349454707),
 (('submarine-ball', 'hurler'), 20.147176349454707),
 (('6-foot-3-inch', '158-pounder'), 20.147176349454707),
 (("tuttle's", '390-foot'), 20.147176349454707),
 (("forbes's", 'paget'), 20.147176349454707),
 (("hallowell's", 'milties'), 20.147176349454707),
 (('swift-striding', 'jamaican'), 20.147176349454707),
 (('mal', 'whitfield'), 20.147176349454707),
 (("tech's", 'sweat-suits'), 20.147176349454707),
 (("raiders'", '38-7'), 20.147176349454707),
 (('abner', 'haynes'), 20.147176349454707),
 (('patti', 'waggin'), 20.147176349454707),
 (('camilo', 'carreon'), 20.147176349454707)]

Observations:
- Top PMI pairs have very high identical scores, High PMI pairs are mostly named entities or fixed phrases which strongly co-occur
- This shows PMI’s bias toward low-frequency events.# - High PMI pairs are mostly named entities or fixed phrases#   (e.g., "mal whitfield"), which strongly co-occur.
- Low PMI pairs involve frequent function wordsshowing weak or no association.
- PPMI sets negative PMI values to zero, keeping only meaningful associations resulting in a cleaner representation.

In [22]:
with open('brown_100.txt', 'r') as file_:
    corpus = file_.read()

In [40]:
import re
clean_text = re.sub(r'[^a-z\s]', '', corpus, flags=re.IGNORECASE)

words = word_tokenize(clean_text)
words = [word.lower() for word in words]


stop_words = set(stopwords.words('english'))
words = [word for word in words if word not in stop_words]

In [45]:
word_counts = Counter(tokens)

vocab = {w for w in word_counts if word_counts[w] >= 10}

filtered_words = [w for w in words if w in vocab]
bigram_counts = Counter(zip(filtered_words[:-1], filtered_words[1:]))

N = sum(word_counts.values())


unigram_counts = Counter(words)
bigram_counts = Counter(zip(words, words[1:]))

In [46]:
def compute_pmi(w1, w2):
    p_w1 = word_counts[w1] / N
    p_w2 = word_counts[w2] / N
    p_w1_w2 = bigram_counts[(w1, w2)] / (N - 1)
    
    return math.log2(p_w1_w2 / (p_w1 * p_w2))

def compute_ppmi(w1, w2):
    pmi = compute_pmi(w1, w2)
    return max(0, pmi)


In [47]:
pmi_scores = []
for (w1, w2), count in bigram_counts.items():
    if count > 0:
        pmi_scores.append(((w1, w2), compute_pmi(w1, w2)))

pmi_sorted = sorted(pmi_scores, key=lambda x: x[1], reverse=True)

In [48]:
top_20 = pmi_sorted[:20]
top_20


[(('produced', 'evidence'), 10.184876409541907),
 (('termend', 'presentments'), 10.184876409541907),
 (('deserves', 'praise'), 10.184876409541907),
 (('praise', 'thanks'), 10.184876409541907),
 (('conducted', 'septemberoctober'), 10.184876409541907),
 (('judge', 'durwood'), 10.184876409541907),
 (('durwood', 'pye'), 10.184876409541907),
 (('pye', 'investigate'), 10.184876409541907),
 (('relative', 'handful'), 10.184876409541907),
 (('considering', 'widespread'), 10.184876409541907),
 (('outmoded', 'inadequate'), 10.184876409541907),
 (('inadequate', 'often'), 10.184876409541907),
 (('often', 'ambiguous'), 10.184876409541907),
 (('studied', 'revised'), 10.184876409541907),
 (('revised', 'end'), 10.184876409541907),
 (('end', 'modernizing'), 10.184876409541907),
 (('modernizing', 'improving'), 10.184876409541907),
 (('topics', 'among'), 10.184876409541907),
 (('generally', 'accepted'), 10.184876409541907),
 (('inure', 'best'), 10.184876409541907)]

In [49]:
bottom_20 = pmi_sorted[-20:]
bottom_20

[(('place', 'jury'), 4.014951408099596),
 (('received', 'jury'), 4.014951408099596),
 (('departments', 'said'), 4.014951408099595),
 (('chairman', 'said'), 4.014951408099595),
 (('said', 'also'), 4.014951408099595),
 (('said', 'polls'), 4.014951408099595),
 (('said', 'ordinary'), 4.014951408099595),
 (('resolution', 'would'), 3.8450264066572832),
 (('resolution', 'resolution'), 3.8450264066572832),
 (('said', 'friday'), 3.5999139088207515),
 (('mayor', 'said'), 3.5999139088207515),
 (('petition', 'said'), 3.5999139088207515),
 (('state', 'funds'), 3.5999139088207515),
 (('funds', 'state'), 3.5999139088207515),
 (('department', 'fulton'), 3.2075964860419917),
 (('election', 'city'), 3.2075964860419917),
 (('city', 'jury'), 2.8450264066572832),
 (('department', 'jury'), 2.8450264066572832),
 (('said', 'resolution'), 2.4299889073784393),
 (('said', 'state'), 2.0149514080995954)]

In [50]:
ppmi_scores = []
for (w1, w2), count in bigram_counts.items():
    if count > 0:
        ppmi_scores.append(((w1, w2), compute_ppmi(w1, w2)))

ppmi_sorted = sorted(ppmi_scores, key=lambda x: x[1], reverse=True)

In [51]:
top_20_ppmi = ppmi_sorted[:20]
top_20_ppmi

[(('produced', 'evidence'), 10.184876409541907),
 (('termend', 'presentments'), 10.184876409541907),
 (('deserves', 'praise'), 10.184876409541907),
 (('praise', 'thanks'), 10.184876409541907),
 (('conducted', 'septemberoctober'), 10.184876409541907),
 (('judge', 'durwood'), 10.184876409541907),
 (('durwood', 'pye'), 10.184876409541907),
 (('pye', 'investigate'), 10.184876409541907),
 (('relative', 'handful'), 10.184876409541907),
 (('considering', 'widespread'), 10.184876409541907),
 (('outmoded', 'inadequate'), 10.184876409541907),
 (('inadequate', 'often'), 10.184876409541907),
 (('often', 'ambiguous'), 10.184876409541907),
 (('studied', 'revised'), 10.184876409541907),
 (('revised', 'end'), 10.184876409541907),
 (('end', 'modernizing'), 10.184876409541907),
 (('modernizing', 'improving'), 10.184876409541907),
 (('topics', 'among'), 10.184876409541907),
 (('generally', 'accepted'), 10.184876409541907),
 (('inure', 'best'), 10.184876409541907)]

# Observation:
- Top PMI pairs like ('produced', 'evidence'), ('deserves', 'praise'), and ('outmoded', 'inadequate') are meaningful word combinations. Unlike the full Brown corpus, these pairs are less noisy and not dominated by rare proper nouns, which makes the results more interpretable.
- Bottom PMI pairs like ('said', 'state') or ('city', 'jury') still have positive values, meaning even the weakest pairs here co-occur more than expected.
- PPMI results are identical to PMI because there are no negative PMI values, so nothing gets clipped to zero
